# <b>Preprocess Data</b>
The cleaned matrix is subjected to normalization of UMI counts per cell, followed by the selection of Highly Variable Genes (HVGs).

---

## 1. Setup Environment

### 1.1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import scanpy as sc

### 1.2. Define File Paths/Directories

In [ ]:
import config

# Cleaned AnnData file
CLEANED_ANNDATA_FILE = config.PROCESSED_DATA_DIR / "01_Norman_2019_cleaned.h5ad"
print("Cleaned AnnData file:\n", CLEANED_ANNDATA_FILE)

# Preprocessed AnnData file save directory
PREPROCESSED_ANNDATA_FILE = config.PROCESSED_DATA_DIR / "02_Norman_2019_preprocessed.h5ad"
print("\nPreprocessed AnnData file will be saved to:\n", PREPROCESSED_ANNDATA_FILE)

# Figures save directory
FIGURES_DIR = config.FIGURES_DIR / "02_preprocessing"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print("\nFigures will be saved to:\n", FIGURES_DIR)

### 1.3. Define Global Parameters

In [ ]:
SEED = config.SEED

np.random.seed(SEED)
sc.settings.seed = SEED

print(f"\nRandom seed set to: {SEED}")

---

## 2. Load And Validate Data

### 2.1. Load Cleaned AnnData

In [ ]:
# Read the cleaned AnnData file
print("\nReading cleaned AnnData file...")
adata = sc.read_h5ad(CLEANED_ANNDATA_FILE)
print(adata)

### 2.2. Inspect Data

In [ ]:
print("Cell IDs:\n", adata.obs_names)
print("\nGene IDs:\n", adata.var_names)

print("\nGene symbols:\n", adata.var.gene_symbols)

print("\nSample data matrix (first 5 rows and columns):\n", adata.X[:5, :5].toarray())

### 2.3. Validate Data

#### 2.3.1. Verify Raw UMI Counts
Check if the data entirely consists of whole numbers, therefore proving that it contains raw UMI counts.

In [ ]:
is_raw_counts = np.all(adata.X.data == np.floor(adata.X.data))

if is_raw_counts:
    print("\nData matrix contains raw UMI counts.")
else:
    print("\nFAILED: Data matrix does NOT contain raw UMI counts.")

#### 2.3.2. Index Genes By Symbol
Index genes by their symbols instead of Ensembl IDs.

In [ ]:
# Check if genes are already indexed by their symbols
are_genes_ens_ids = adata.var_names.str.startswith("ENSG").all()

# Map to gene symbols if currently indexed by Ensembl IDs
if are_genes_ens_ids:
    print("\nGenes are indexed by Ensembl IDs. Mapping to gene symbols....")

    # Check if gene symbols are unique
    if adata.var.gene_symbols.is_unique:
        print("Gene symbols are unique. Setting gene symbols as index....")
        adata.var_names = adata.var['gene_symbols'].values
        print("\nGene symbols set as index successfully.")
    else:
        print("\nGene symbols are NOT unique.")

        # How many duplicates?
        duplicates = adata.var['gene_symbols'][adata.var['gene_symbols'].duplicated(keep=False)]
        print(f"Number of duplicate gene symbol entries: {len(duplicates)}")
        print(duplicates.value_counts().head())

        # Make gene symbols unique by appending a suffix to duplicates
        print("\nMaking gene symbols unique and setting them as index....")
        adata.var_names = adata.var['gene_symbols'].values
        adata.var_names_make_unique()
        print("Gene symbols made unique and set as index successfully.")

# If genes are already indexed by gene symbols, no mapping is needed
else:
    print("\nGenes are already indexed by gene symbols. No mapping needed.")

print("\nGene IDs:\n", adata.var_names)

#### 2.3.4. Verify Uniqueness Of Cell IDs

In [ ]:
are_cell_ids_unique = adata.obs_names.is_unique

if are_cell_ids_unique:
    print("\nCell IDs are unique.")
else:
    print("\nFAILED: Cell IDs are NOT unique.")

---

## 3. Normalize Data

### 3.1. Stash Raw UMI Counts Before Normalization

In [ ]:
print("Stashing raw counts in adata.layers['counts']...")
adata.layers['counts'] = adata.X.copy()
print("\nadata.layers['counts'] set to a copy of the raw counts matrix. Inspecting values in adata.layers['counts']:")
print(adata.layers['counts'][:5, :5].toarray())

### 3.2. Perform CP10K Normalization
Normalize UMI counts of all genes in all cells such that the total UMI count of all genes in each cell is `10,000`.

In [ ]:
# Normalize each cell to 10,000 counts
print("\nNormalizing each cell to 10,000 counts (CP10K normalization)...")
sc.pp.normalize_total(adata, target_sum=1e4)
print("CP10K normalization complete.")

# Inspect total counts per cell
cell_sums = np.array(adata.X.sum(axis=1)).flatten()
print(f"Cell total counts after normalization (should all be ~10000):")
print(
    f"Min: {cell_sums.min():.1f}, Max: {cell_sums.max():.1f}, Mean: {cell_sums.mean():.1f}"
)

print("\nInspecting first 5 rows and columns of normalized data matrix (should now have floats):\n", adata.X[:5, :5].toarray())

### 3.3. Apply Log1P Transformation To CP10K Normalized Data

In [ ]:
# Apply log1p transformation to the normalized data
print("\nApplying log1p transformation to normalized data...")
sc.pp.log1p(adata)
print("log1p transformation complete.")

# Inspect the data matrix after log1p transformation
sample_values = adata.X[:5, :5].toarray()
print(
    "Inspecting first 5 rows and columns after log1p (should have lesser float values now):"
)
print(sample_values)

## 4. Select Highly Variable Genes (HVGs)

In [ ]:
# Select the top N HVGs
top_n_hvgs = 3000  # Iteratively adjusted

# Identify highly variable genes
print(
    f"\nIdentifying the top {top_n_hvgs} highly variable genes using the 'seurat_v3' method..."
)
sc.pp.highly_variable_genes(
    adata, flavor="seurat_v3", n_top_genes=top_n_hvgs, layer="counts"
)
print("Highly variable gene identification complete.")

# Plot and save
fig = sc.pl.highly_variable_genes(adata, show=False)
plt.savefig(str(FIGURES_DIR / "01_hvgs.png"), bbox_inches="tight")
plt.show()

print(f"Total HVGs selected: {adata.var['highly_variable'].sum()}")

---

## 5. Perform Final Sanity Checks

In [ ]:
print("=== POST-PREPROCESSING SANITY CHECK ===\n")

# Verify normalization
x_max = adata.X.data.max()
x_min = adata.X.data.min()
assert x_min >= 0, "Negative values found — normalization error"
assert (
    x_max <= np.log1p(1e4) + 0.01
), f"Max value {x_max:.3f} exceeds log1p(10000) — log1p may not have been applied"
print(f"Normalization and log1p verified (value range: {x_min:.3f} - {x_max:.3f})")

# Verify raw counts layer preserved
assert "counts" in adata.layers, "Raw counts layer missing"
assert np.all(
    adata.layers["counts"].data == np.floor(adata.layers["counts"].data)
), "Raw counts layer contains non-integers"
print("Raw counts layer preserved")

# Verify HVGs selected
assert adata.var["highly_variable"].sum() == 3000, "HVG count mismatch"
print("HVGs selected: 3000")

print("\nAll assertions passed.")

---

## 6. Save Cleaned AnnData

In [ ]:
adata.write_h5ad(PREPROCESSED_ANNDATA_FILE)
print(f"Saved: {PREPROCESSED_ANNDATA_FILE}")

---

## 7. Summary

### 7.1. Data Validation
- Gene symbols confirmed as index (no Ensembl IDs)
- Cell barcodes confirmed unique
- Raw UMI counts confirmed as integers

### 7.2. Steps Executed
1. **Raw count stashing** — UMI counts copied to `adata.layers['counts']` before any normalization
2. **CP10K normalization** — each cell scaled to 10,000 total counts to remove sequencing depth differences
3. **log1p transformation** — compresses dynamic range and stabilizes mean-variance relationship
4. **HVG selection** — top 3,000 highly variable genes identified using seurat_v3 on raw counts layer

### 7.3. Parameters
- Normalization target: 10,000 counts per cell
- HVG method: seurat_v3 (uses raw counts layer)
- Number of HVGs: 3,000

### 7.4. AnnData Modifications
**`adata.layers` (named copies of the expression matrix):**
- `counts` — raw UMI counts preserved before normalization

**`adata.X` (main expression matrix):**
- Overwritten with CP10K normalized, log1p transformed values

**`adata.var` (gene-level metadata):**
- `highly_variable` — boolean flag, True for top 3,000 HVGs
- `highly_variable_rank` — rank by normalized variance
- `means`, `variances`, `variances_norm` — per-gene statistics from seurat_v3